[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Transactions


## What you will be able to do

Put several writes in one transaction with `db.atomic`, as a block or as a decorator, and say
exactly which writes a rollback undoes. Explain why an exception you caught inside the block leaves
the work committed, and write the version that does what you meant. Nest `atomic` to get a
savepoint, read the `SAVEPOINT` and `ROLLBACK TO SAVEPOINT` statements it sends, and say what a
savepoint can and cannot survive. Keep a side effect out of the transaction with `after_commit`.
Say what a rollback does not undo, and what the same swallowed exception would do on PostgreSQL.


## The idea

### The problem

`db.atomic()` is a context manager, and a context manager learns that something went wrong by being
handed the exception on the way out. That is the whole mechanism. If you catch an exception inside
the block, nothing is handed to it, the block ends the way a successful block ends, and the
transaction commits.

So the code that looks most careful is the code that fails. A loop that writes rows and catches
errors as it goes, wrapped in `atomic` for safety, commits whatever it managed to write before the
error and reports success. The word "atomic" promises all or nothing, and it keeps that promise: it
is just that "nothing went wrong" is defined as "no exception left this block".

### What a transaction is

A run of statements the database treats as one unit. It begins, statements happen, and it ends
either with a commit, which makes all of them permanent at once, or with a rollback, which discards
all of them. Until it ends, nobody else sees any of it.

### Why it works that way

Python has no way to tell a context manager that an exception happened and was dealt with. `__exit__`
receives the exception only when it is propagating. An exception you caught is, as far as the
language is concerned, handled and gone. peewee could not detect the case if it wanted to.

This is the same shape as the filter in the **Selecting Rows** notebook: the mistake is legal
Python that means something other than it reads, and no library can raise on your behalf.

### Where this shows up

Any import or sync that writes several rows and tries to be forgiving about bad ones. Anything that
writes a parent row and then child rows. Anything that sends mail, writes to a queue or calls
another service next to a database write, which is where `after_commit` belongs.

### What this notebook covers

`atomic` as a block and as a decorator. Nesting, which gives a savepoint rather than a new
transaction, with the statements it sends printed. What a rollback undoes in the database and what
it leaves alone in Python. Rolling back deliberately. `after_commit` for work that must not happen
unless the transaction did. The one difference that matters on PostgreSQL. Then the four failures,
three of them silent.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, IntegrityError, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Tag(Model):
    name = CharField(unique=True)

    class Meta:
        database = db


db.create_tables([Tag])

with db.atomic():                               # everything in here is one transaction
    Tag.create(name="fiction")
    try:
        Tag.create(name="fiction")              # the same name twice
    except IntegrityError as error:
        print("caught:", error)

print("rows after the block:", [tag.name for tag in Tag.select()])
```

```
caught: UNIQUE constraint failed: tag.name
rows after the block: ['fiction']
```

The error was caught, so the block ended normally, so the transaction committed, so the first row is
still there. If what you wanted was both tags or neither, this code gave you neither of those
outcomes and told you nothing.


## Setup

Six imports, peewee installed and pinned, the catalog's models, and a database that records.

- `peewee` is the library, and `Model`, the field classes and `SqliteDatabase`, from it, are what a
  model is written with
- `IntegrityError` is the error a duplicate or a missing required value raises, caught many times
  below
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `re` masks the savepoint names, which are fresh on every run
- `AUTHORS` and `BOOKS` are the catalog, and `build` makes the tables and loads them

`db` is a `RecordingSqlite`, which keeps every statement peewee sends so that the savepoint
statements can be printed rather than described. One thing it cannot show: peewee sends `BEGIN` and
`COMMIT` on the connection directly rather than through this method, so those two never appear in
the list. Everything a savepoint does goes through it, which is the part worth seeing.


In [1]:
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, IntegrityError, Model,
                    SqliteDatabase)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

class RecordingSqlite(SqliteDatabase):
    """A database that keeps every statement sent through it, with savepoint names masked.

    peewee names each savepoint with a fresh uuid4, so the names differ on every run. They are
    replaced with s... here so that this notebook prints the same thing each time it is run.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statements = []

    def execute_sql(self, sql, params=None):
        self.statements.append(re.sub(r'"s[0-9a-f]{32}"', '"s..."', " ".join(sql.split())))
        return super().execute_sql(sql, params)


def marks(database):
    """Only the savepoint statements, which is what the nesting looks like from the database."""
    return [line for line in database.statements
            if line.split()[0] in ("SAVEPOINT", "RELEASE", "ROLLBACK")]

db = RecordingSqlite(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books")


peewee 4.5.1 | the catalog: 4 authors and 12 books


## Worked examples

### atomic as a block

Two writes, one transaction. When the block ends normally the pair is committed together:


In [2]:
with db.atomic():
    marco = Author.get(Author.name == "Marco Pietra")
    Book.create(title="The Second Bridge", author=marco, year=2024, pages=210)
    Book.create(title="The Third Bridge", author=marco, year=2025, pages=198)

print("books:", Book.select().count())


books: 14


When an exception leaves the block, neither of them happened:


In [3]:
before = Book.select().count()
try:
    with db.atomic():
        Book.create(title="The Fourth Bridge", author=marco, year=2026, pages=180)
        raise RuntimeError("something after the write went wrong")
except RuntimeError as error:
    print("the error left the block:", error)

print("books before:", before, "| after:", Book.select().count())


the error left the block: something after the write went wrong
books before: 14 | after: 14


The write ran. The database had the row. Then the exception reached `__exit__`, peewee rolled the
transaction back, and the row was discarded.

### atomic as a decorator

The same object works as a decorator, so a whole function can be one transaction. Note the
parentheses on `db.atomic()`: they are not optional, and leaving them off is one of the Common
errors below.


In [4]:
@db.atomic()
def shelve(titles, author, year):
    """Write several books, or none of them."""
    for title in titles:
        Book.create(title=title, author=author, year=year, pages=200)


shelve(["North Light", "South Light"], marco, 2024)
print("books:", Book.select().count())

try:
    shelve(["East Light", None], marco, 2024)                       # None is not a title
except IntegrityError as error:
    print("raised:", error)
print("books:", Book.select().count(), "| East Light written:",
      Book.get_or_none(Book.title == "East Light") is not None)


books: 16
raised: NOT NULL constraint failed: book.title
books: 16 | East Light written: False


`East Light` was written and then taken away again, because the failure on the next line left the
function and so left the transaction.

### A nested atomic is a savepoint

Writing `atomic` inside `atomic` does not start a second transaction. It marks a point inside the
one that is already running, which the database can be told to go back to:


In [5]:
db.statements.clear()
with db.atomic() as outer:
    with db.atomic() as inner:
        pass

print("outer:", type(outer).__name__, "| inner:", type(inner).__name__)
for line in marks(db):
    print("  ", line)


outer: _transaction | inner: _savepoint
   SAVEPOINT "s...";
   RELEASE SAVEPOINT "s...";


Now the part that matters. The same failing write, caught at the same place, with and without an
inner `atomic` around it:


In [6]:
def attempt(use_savepoint):
    """Write one good row, then two rows that collide, and report what survived."""
    Author.delete().where(Author.name.startswith("Test")).execute()
    db.statements.clear()
    with db.atomic():
        Author.create(name="Test outer", first_book=2000)
        try:
            if use_savepoint:
                with db.atomic():                                   # a savepoint
                    Author.create(name="Test inner", first_book=2001)
                    Author.create(name="Test inner", first_book=2002)
            else:
                Author.create(name="Test inner", first_book=2001)
                Author.create(name="Test inner", first_book=2002)
        except IntegrityError:
            pass
    return sorted(a.name for a in Author.select().where(Author.name.startswith("Test")))


print("with a savepoint:   ", attempt(True))
print("  ", marks(db))
print("without a savepoint:", attempt(False))
print("  ", marks(db))


with a savepoint:    ['Test outer']
   ['SAVEPOINT "s...";', 'ROLLBACK TO SAVEPOINT "s...";']
without a savepoint: ['Test inner', 'Test outer']
   []


With the savepoint, the inner work was undone as a unit and the outer row survived. Without it, the
row that had been written before the collision stayed, because nothing told the database to go back
anywhere. The `ROLLBACK TO SAVEPOINT` in the first log is the whole difference.

A savepoint is not a transaction, though. It lives inside one, and if the transaction it lives in
rolls back, everything the savepoint released goes with it. That is one of the Common errors below.

### What a rollback does not undo

The database went back. Python did not:


In [7]:
held = None
try:
    with db.atomic():
        held = Author.create(name="Never Written", first_book=1900)
        raise RuntimeError("stop")
except RuntimeError:
    pass

print("in the database:", Author.get_or_none(Author.name == "Never Written"))
print("the instance still says:", repr(held.name), "| and still has id:", held.id)


in the database: None
the instance still says: 'Never Written' | and still has id: 7


The instance is an ordinary Python object holding the values you gave it and the `id` the database
handed back before the rollback. peewee has no session watching it, as the **Creating and Changing
Rows** notebook explained, so there is nothing to reset. An instance that survives a rollback is a
lie about the database, and saving it later writes a row with an `id` that may now belong to
something else. Let go of your instances when a transaction fails.

### Rolling back on purpose

The block object has `rollback` and `commit`, for deciding partway through:


In [8]:
with db.atomic() as transaction:
    Author.create(name="Discarded", first_book=1800)
    transaction.rollback()                                          # everything above is gone
    Author.create(name="Kept", first_book=1801)

for name in ("Discarded", "Kept"):
    print(f"  {name:<10} in the database: {Author.get_or_none(Author.name == name) is not None}")
Author.delete().where(Author.name == "Kept").execute()


  Discarded  in the database: False
  Kept       in the database: True


1

`rollback` discards the work so far and starts a new transaction in its place, so the block carries
on and the writes after it are committed at the end. That is useful and it is also a trap worth
naming: the block did not stop.

### Work to do only if the transaction commits

Sending mail, writing to a queue or calling another service are not part of the transaction and
cannot be rolled back. Done inside the block, they happen whether or not the write survives:


In [9]:
sent = []

try:
    with db.atomic():
        author = Author.create(name="Rolled Back", first_book=1700)
        sent.append(f"welcome message for {author.name}")           # happens immediately
        raise RuntimeError("a later write failed")
except RuntimeError:
    pass

print("in the database:", Author.get_or_none(Author.name == "Rolled Back"))
print("side effects that happened anyway:", sent)


in the database: None
side effects that happened anyway: ['welcome message for Rolled Back']


`after_commit` takes a callable and runs it only when the outermost transaction commits. On a
rollback it is discarded:


In [10]:
sent.clear()

try:
    with db.atomic():
        author = Author.create(name="Rolled Back", first_book=1700)
        db.after_commit(lambda: sent.append(f"welcome message for {author.name}"))
        raise RuntimeError("a later write failed")
except RuntimeError:
    pass
print("after a rollback:", sent)

with db.atomic():
    author = Author.create(name="Committed", first_book=1701)
    db.after_commit(lambda: sent.append(f"welcome message for {author.name}"))
    print("inside the block:", sent)
print("after the block:  ", sent)
Author.delete().where(Author.name == "Committed").execute()


after a rollback: []
inside the block: []
after the block:   ['welcome message for Committed']


1

Inside the block the list is still empty, which is the point: the callable runs after the commit,
not at the line that registered it.

### The same code on PostgreSQL

Everything above was run on SQLite, and one behavior would be different on PostgreSQL. When a
statement fails there, the transaction is put into an aborted state, and every statement after it
fails too, with `current transaction is aborted, commands ignored until end of transaction block`,
until the transaction ends. So the swallowed `IntegrityError` that quietly commits half a batch on
SQLite instead turns every later statement in the block into an error on PostgreSQL.

That makes the savepoint more than a tidiness measure there: wrapping each risky write in a nested
`atomic` is what lets a loop carry on after one row fails, because rolling back to the savepoint is
what clears the aborted state. Code written and tested on SQLite without savepoints can therefore
fail on PostgreSQL in a way it never failed locally. The **SQLite and PostgreSQL** notebook is where
the two backends are compared, and the message above is quoted there and in **asyncpg and psycopg3,
Deep Dive** rather than reproduced here, because this guide does not run a PostgreSQL server.

### When to reach for which

| What you want | How to write it |
|---|---|
| several writes that must all happen | `with db.atomic():` |
| a whole function as one transaction | `@db.atomic()` above it |
| one risky write inside a longer transaction | a nested `with db.atomic():` |
| to discard the work so far and carry on | `transaction.rollback()` |
| to keep the work so far and carry on | `transaction.commit()` |
| a side effect that must not happen on rollback | `db.after_commit(callable)` |
| to know what the nesting sent | the `SAVEPOINT` lines in `db.statements` |

The default is the plain block: open one `atomic` around the unit of work and let exceptions leave
it. Reach for a nested `atomic` only where you intend to carry on after a failure, and for
`rollback` and `commit` by hand only where the decision depends on something you learn partway
through.

### A safe import, finished

An import that writes a parent row and its children together, skips a record the database refuses
without losing the rest, and queues its notifications so that none are sent for work that did not
survive.


In [11]:
def import_author(name, first_book, titles):
    """Write an author and their books as one unit, and report what happened."""
    with db.atomic():                                               # all of it, or none of it
        author = Author.create(name=name, first_book=first_book)
        written, skipped = [], []
        for title, year, pages in titles:
            try:
                with db.atomic():                                   # one savepoint per book
                    Book.create(title=title, author=author, year=year, pages=pages)
                written.append(title)
            except IntegrityError:
                skipped.append(title)
        db.after_commit(lambda: queued.append(f"{name}: {len(written)} books"))
        return written, skipped


queued = []
books = [("Low Tide", 2021, 240), (None, 2022, 260), ("High Tide", 2023, 190)]
print("written, skipped:", import_author("Wole Adeyemi", 2021, books))
print("queued:", queued)

try:
    import_author("Wole Adeyemi", 2021, books)                      # the author already exists
except IntegrityError as error:
    print("second run raised:", error)
print("authors named Wole:", Author.select().where(Author.name == "Wole Adeyemi").count())
print("queued:", queued)


written, skipped: (['Low Tide', 'High Tide'], [None])
queued: ['Wole Adeyemi: 2 books']
second run raised: UNIQUE constraint failed: author.name
authors named Wole: 1
queued: ['Wole Adeyemi: 2 books']


The record with no title was refused, and its savepoint took that one book back without touching
the two good ones or the author. The second run failed on the author itself, before any book, so
nothing was written and nothing was queued. The notification list is the evidence: one entry, for
the one run that committed.

### Where each part came from

| In the import | What it relies on | The section that showed it |
|---|---|---|
| `with db.atomic():` around the whole function | one unit that commits or does not | atomic as a block |
| a nested `with db.atomic():` per book | a savepoint one book can fail inside | A nested atomic is a savepoint |
| `except IntegrityError` around the inner block | catching outside the savepoint, not inside it | A nested atomic is a savepoint |
| `db.after_commit(...)` | a side effect the rollback can discard | Work to do only if it commits |
| letting the author's error leave the function | the rollback that undoes everything | atomic as a block |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/05-transactions-solutions.ipynb).

**1.** Write two books in one `atomic` block, raise an error after the second, and show that neither
was kept.


In [12]:
# your code here


**2.** Write the same pair inside a block that catches the error instead, and show what survives.


In [13]:
# your code here


**3.** Write a function decorated with `@db.atomic()` that writes an author and one book, and show
that a failure in the book undoes the author.


In [14]:
# your code here


**4.** Write a loop over four author names, two of them duplicates, with a savepoint around each
one, so that the good names are kept and the collisions are skipped. Print the savepoint statements.


In [15]:
# your code here


**5.** Register two `after_commit` callables in one block and show the order they run in, and that
neither runs before the block ends.


In [16]:
# your code here


**6.** Hold an instance through a rollback and print three things: what the instance says its name
is, what `id` it has, and whether that row is in the database.


In [17]:
# your code here


## Common errors

### No error, and half the batch kept: an exception caught inside the atomic block


In [18]:
Author.delete().where(Author.name.startswith("Batch")).execute()

with db.atomic():
    for number, name in enumerate(["Batch one", "Batch two", "Batch two", "Batch three"]):
        try:
            Author.create(name=name, first_book=1900 + number)
        except IntegrityError:
            print("  skipped a duplicate:", name)

print("kept:", sorted(a.name for a in Author.select().where(Author.name.startswith("Batch"))))


  skipped a duplicate: Batch two
kept: ['Batch one', 'Batch three', 'Batch two']


Three rows committed, out of a block that was written to be all or nothing. Every `except` inside an
`atomic` block is a decision to commit whatever came before it, whether or not that was the
intention.

Two fixes, and which one is right depends on what you meant. If the batch really is all or nothing,
do not catch the error at all, and let it leave the block. If bad rows should be skipped and the
rest kept, say so with a savepoint around each one, so that the skipping is a real rollback of that
row rather than an absence of one:


In [19]:
Author.delete().where(Author.name.startswith("Batch")).execute()
db.statements.clear()

with db.atomic():
    for number, name in enumerate(["Batch one", "Batch two", "Batch two", "Batch three"]):
        try:
            with db.atomic():                                       # this row, on its own
                Author.create(name=name, first_book=1900 + number)
        except IntegrityError:
            pass

print("kept:", sorted(a.name for a in Author.select().where(Author.name.startswith("Batch"))))
print("one rollback per bad row:", marks(db).count("ROLLBACK TO SAVEPOINT \"s...\";"))


kept: ['Batch one', 'Batch three', 'Batch two']
one rollback per bad row: 1


### No error, and a message sent about a row that does not exist: a side effect inside the block


In [20]:
messages = []

try:
    with db.atomic():
        author = Author.create(name="Never Kept", first_book=1600)
        messages.append(f"welcome {author.name}")
        Author.create(name="Never Kept", first_book=1601)           # collides, and is not caught
except IntegrityError:
    pass

print("in the database:", Author.get_or_none(Author.name == "Never Kept"))
print("messages sent:", messages)


in the database: None
messages sent: ['welcome Never Kept']


The row is gone and the message went out. A transaction can roll back a write to its own database
and nothing else, so anything that reaches outside has to wait until the transaction is known to
have committed. `db.after_commit` is where that work goes.

### No error, and the inner block's rows gone: a savepoint is not a transaction


In [21]:
Author.delete().where(Author.name == "Inner Done").execute()

try:
    with db.atomic():
        with db.atomic():
            Author.create(name="Inner Done", first_book=1500)
        print("  the inner block finished with no error")
        raise RuntimeError("the outer block fails later")
except RuntimeError:
    pass

print("in the database:", Author.get_or_none(Author.name == "Inner Done"))


  the inner block finished with no error
in the database: None


The inner block ended cleanly and released its savepoint, and the row still went away, because
releasing a savepoint only means the outer transaction now owns that work. Nothing is durable until
the outermost block commits. A nested `atomic` gives you a unit that can fail on its own, not a unit
that can succeed on its own.

### TypeError: _callable_context_manager.__call__() missing 1 required positional argument: 'fn'


In [22]:
@db.atomic
def write_one():
    Author.create(name="Never Runs", first_book=1400)


write_one()


TypeError: _callable_context_manager.__call__() missing 1 required positional argument: 'fn'

`db.atomic` is a method. `db.atomic()` is the object that works as a context manager and as a
decorator. Without the parentheses the decorator syntax calls the method with your function as its
first argument, so `write_one` is now a context manager rather than a function, and calling it asks
for the function it was never given.

Note where it raised: the decoration itself was accepted, and the error waited for the call. A
module that defines such a function imports cleanly.

The `with` form has the same rule, and fails more clearly there, since `with db.atomic:` has nothing
to enter. With the parentheses:


In [23]:
@db.atomic()
def write_one():
    Author.create(name="Now Runs", first_book=1400)


write_one()
print("written:", Author.get_or_none(Author.name == "Now Runs") is not None)


written: True


## Recap

- `db.atomic()` makes one transaction. It commits when the block ends normally and rolls back when
  an exception leaves it.
- An exception you catch inside the block never leaves it, so the block ends normally and the work
  commits. This is the most common way to lose a transaction's guarantee.
- `@db.atomic()` decorates a function as one transaction. The parentheses are required.
- A nested `atomic` is a savepoint. It can be rolled back on its own, and it is not durable on its
  own: if the outer transaction rolls back, the savepoint's work goes too.
- A rollback undoes the database and nothing else. The instance you are holding keeps its values and
  its `id`, and neither is true any more.
- `transaction.rollback()` discards the work so far and lets the block carry on.
- `db.after_commit(callable)` runs work only if the outermost transaction commits, which is where
  mail, queues and calls to other services belong.
- On PostgreSQL a failed statement aborts the whole transaction until it ends, so a savepoint around
  each risky write is what lets a loop carry on there.


## What is next

The **Relationships** notebook turns to the foreign key: `ForeignKeyField` and the `backref` it
creates, what `author.books` and `book.author` each cost, joins written with `join` and read back
with `objects`, and the query that looks like one join and sends one statement per row.


---

&#8592; **Previous:** [Selecting Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/04-selecting-rows.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
